
# 08 — Sparkov Final Held-Out Test: M1 CatBoost

This notebook evaluates the **selected M1 static cost-sensitive CatBoost model** on the held-out test set.

M1 achieved the highest validation PR-AUC (**0.989228**) among the evaluated models and was therefore selected for final testing.

The final evaluation reports PR-AUC, ROC-AUC, precision, recall, and operational performance under daily alert limits. Model selection and operational settings are determined using validation data, while the held-out test set is used only to assess final performance.


## 1. Load data, locked M1 predictions and verify chronological alignment

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score

DATA_DIR = Path('../data/processed')
M3_RESULTS_DIR = Path('../results/m3_catboost_selected_m1')
M4_RESULTS_DIR = Path('../results/m4_catboost_isolation_forest')
RESULTS_DIR = Path('../results/sparkov_final_test_catboost')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TIME_COL, TARGET_COL, SPLIT_COL = 'trans_date_trans_time', 'is_fraud', 'sparkov_split'

frame = pd.read_pickle(DATA_DIR / 'sparkov_features.pkl')
with open(DATA_DIR / 'feature_lists.json') as f:
    feature_cols = json.load(f)['sparkov_feature_cols']
frame[TIME_COL] = pd.to_datetime(frame[TIME_COL])
# Preserve original indices: Notebook 07/08 prediction row_index references these.
frame = frame.loc[frame[SPLIT_COL].isin(['train', 'validation', 'test'])].copy()
train_rows = frame.loc[frame[SPLIT_COL].eq('train')].copy()
val_rows = frame.loc[frame[SPLIT_COL].eq('validation')].copy()
test_rows = frame.loc[frame[SPLIT_COL].eq('test')].copy()
assert len(train_rows) and len(val_rows) and len(test_rows)
assert train_rows[TIME_COL].max() < val_rows[TIME_COL].min() < val_rows[TIME_COL].max() < test_rows[TIME_COL].min(), 'Splits overlap chronologically'
assert 'amt' in frame.columns, 'Missing raw transaction amount for operational costs'
assert not frame.index.has_duplicates, 'Unique original row indices required'

selected_file = M4_RESULTS_DIR / 'selected_stage_after_m4.csv'
if not selected_file.exists():
    raise FileNotFoundError(f'Missing locked M4 selection: {selected_file}')
selection = pd.read_csv(selected_file).iloc[0]
assert str(selection['selected_stage']).upper() == 'M1', f"Expected M1 winner; found {selection['selected_stage']}"

m1_file = M3_RESULTS_DIR / 'm1_catboost_validation_predictions.csv'
if not m1_file.exists():
    raise FileNotFoundError(f'Missing M1 validation predictions: {m1_file}')
m1_val = pd.read_csv(m1_file)
assert {'row_index','y_true','y_score'}.issubset(m1_val.columns)
assert m1_val['row_index'].is_unique and len(m1_val) == len(val_rows)
assert set(m1_val['row_index'].astype(int)) == set(val_rows.index)
m1_val = m1_val.set_index('row_index').loc[val_rows.index].reset_index().rename(columns={'index':'row_index'})
assert np.array_equal(m1_val['y_true'].astype(int).to_numpy(), val_rows[TARGET_COL].astype(int).to_numpy())
if 'timestamp' in m1_val.columns:
    assert np.array_equal(pd.to_datetime(m1_val['timestamp']).to_numpy(), val_rows[TIME_COL].to_numpy())
val_pred = pd.DataFrame({'row_index':val_rows.index.to_numpy(), 'timestamp':val_rows[TIME_COL].to_numpy(),
                         'y_true':val_rows[TARGET_COL].astype(int).to_numpy(),
                         'y_score':m1_val['y_score'].astype(float).to_numpy(),
                         'amount':pd.to_numeric(val_rows['amt'],errors='raise').to_numpy()})
val_pred['date'] = val_pred['timestamp'].dt.floor('D')
val_ap = average_precision_score(val_pred.y_true, val_pred.y_score)
assert np.isclose(val_ap, 0.989228, atol=0.0001), f'Unexpected validation PR-AUC {val_ap:.6f}'
assert val_pred['y_score'].between(0,1).all()
print(f'M1 saved validation PR-AUC: {val_ap:.6f}')
print('Train / validation / test rows:', len(train_rows), len(val_rows), len(test_rows))
print('Test period:', test_rows[TIME_COL].min(), 'to', test_rows[TIME_COL].max())

M1 saved validation PR-AUC: 0.989228
Train / validation / test rows: 1037340 259335 555719
Test period: 2020-06-21 12:14:25 to 2020-12-31 23:59:34


## 2. Lock daily top-k alert capacity using validation only

The review-cost model follows the archived notebook 08_sparkov_final_test_old: **$3.47 per reviewed alert + the sum of transaction amounts for missed frauds**. This assumes a flagged fraud is prevented and a missed fraud incurs its full transaction amount; it is an illustrative cost proxy. Daily top-k ranks every day separately, with deterministic row-index tie-breaking.

In [2]:
REVIEW_COST = 3.47
K_VALUES = [5,10,15,20,25,50,75,100,150,200,300,400,500,750,1000,1500,2000]

def daily_topk_flags(pred, k):
    if k < 0: raise ValueError('k must be nonnegative')
    ranked = pred.sort_values(['date','y_score','row_index'],ascending=[True,False,True])
    selected = ranked.groupby('date',sort=False).cumcount().lt(k)
    return pd.Series(selected.to_numpy(),index=ranked.index).reindex(pred.index).astype(bool)

def operating_metrics(pred, k, review_cost=REVIEW_COST):
    flags = daily_topk_flags(pred,k)
    truth = pred['y_true'].astype(bool)
    tp = int((flags & truth).sum()); fp = int((flags & ~truth).sum())
    fn = int((~flags & truth).sum()); tn = int((~flags & ~truth).sum())
    missed_amount = float(pred.loc[~flags & truth,'amount'].sum())
    review_amount = float(review_cost*(tp+fp))
    baseline_amount = float(pred.loc[truth,'amount'].sum())
    total_cost = review_amount+missed_amount
    return {'k':int(k),'days':int(pred['date'].nunique()),'alerts':tp+fp,
            'tp':tp,'fp':fp,'fn':fn,'tn':tn,
            'precision':tp/(tp+fp) if tp+fp else 0.0,
            'recall':tp/(tp+fn) if tp+fn else 0.0,
            'review_cost':review_amount,'missed_fraud_amount':missed_amount,
            'total_cost':total_cost,'no_alert_cost':baseline_amount,
            'cost_savings':baseline_amount-total_cost,
            'cost_savings_pct':(baseline_amount-total_cost)/baseline_amount*100 if baseline_amount else np.nan}

k_results = pd.DataFrame([operating_metrics(val_pred,k) for k in K_VALUES])
BEST_K = int(k_results.sort_values(['total_cost','k']).iloc[0]['k'])
print(f'LOCKED daily alert capacity: K={BEST_K} (selected on validation ONLY)')
print('Validation transactions/day:')
display(val_pred.groupby('date').size().describe())
display(k_results.sort_values(['total_cost','k']).head(10))
k_results.to_csv(RESULTS_DIR/'sparkov_validation_k_sensitivity.csv',index=False)
pd.DataFrame([{'selected_k':BEST_K,'review_cost_per_alert':REVIEW_COST,
               'validation_pr_auc':val_ap,'selection_split':'validation'}]).to_csv(
                   RESULTS_DIR/'sparkov_selected_alert_policy.csv',index=False)

LOCKED daily alert capacity: K=50 (selected on validation ONLY)
Validation transactions/day:


count     108.000000
mean     2401.250000
std       764.763504
min      1366.000000
25%      1812.000000
50%      2116.000000
75%      3103.500000
max      4167.000000
dtype: float64

,k,days,alerts,tp,fp,fn,tn,precision,recall,review_cost,missed_fraud_amount,total_cost,no_alert_cost,cost_savings,cost_savings_pct
5,50,108,5400,1534,3866,4,253931,0.284074,0.997399,18738.0,852.79,19590.79,816266.27,796675.48,97.599951
6,75,108,8100,1536,6564,2,251233,0.189630,0.998700,28107.0,68.59,28175.59,816266.27,788090.68,96.548235
7,100,108,10800,1537,9263,1,248534,0.142315,0.999350,37476.0,20.62,37496.62,816266.27,778769.65,95.406325
4,25,108,2700,1411,1289,127,256508,0.522593,0.917425,9369.0,40278.30,49647.30,816266.27,766618.97,93.917757
8,150,108,16200,1538,14662,0,243135,0.094938,1.000000,56214.0,0.00,56214.00,816266.27,760052.27,93.113277
9,200,108,21600,1538,20062,0,237735,0.071204,1.000000,74952.0,0.00,74952.00,816266.27,741314.27,90.817702
3,20,108,2160,1314,846,224,256951,0.608333,0.854356,7495.2,73688.93,81184.13,816266.27,735082.14,90.054210
10,300,108,32400,1538,30862,0,226935,0.047469,1.000000,112428.0,0.00,112428.00,816266.27,703838.27,86.226553
11,400,108,43200,1538,41662,0,216135,0.035602,1.000000,149904.0,0.00,149904.00,816266.27,666362.27,81.635404
2,15,108,1620,1155,465,383,257332,0.712963,0.750975,5621.4,145954.03,151575.43,816266.27,664690.84,81.430639


## 3. Refit selected static M1 CatBoost once, then score untouched test

**Final deployment-style refit:** use all **verified pre-test train + validation** labels (both precede test), with the exact selected M1 hyperparameters and feature list. The saved validation predictions remain genuinely out-of-sample from the earlier train-only model. This is **not** weekly retraining, and no tuning or early stopping uses test labels. If comparing the test score with the original M1 validation score, note the final model has more training data.

In [3]:
categorical_cols = train_rows[feature_cols].select_dtypes(include=['object','category','string']).columns.tolist()
assert 'iforest_score' not in feature_cols

def prepare_catboost(rows):
    X = rows[feature_cols].copy()
    for col in categorical_cols:
        X[col] = X[col].fillna('Missing').astype(str)
    return X

# The project's 7-day verification delay also applies at the final test cutoff.
# Labels from the final seven days before test are NOT yet verified.
LABEL_DELAY_DAYS = 7
TEST_CUTOFF = test_rows[TIME_COL].min().floor('D')
VERIFIED_BEFORE = TEST_CUTOFF - pd.Timedelta(days=LABEL_DELAY_DAYS)
fit_rows = frame.loc[
    frame[SPLIT_COL].isin(['train','validation']) &
    frame[TIME_COL].lt(VERIFIED_BEFORE)
].copy()
assert len(fit_rows) > 0, 'No verified training observations at test cutoff'
assert fit_rows[TIME_COL].max() < VERIFIED_BEFORE
assert fit_rows[TIME_COL].max() < test_rows[TIME_COL].min()
print('Test cutoff:', TEST_CUTOFF, '| Verified labels strictly before:', VERIFIED_BEFORE)
print('Excluded pre-test unverified rows:',
      len(frame.loc[frame[SPLIT_COL].isin(['train','validation'])]) - len(fit_rows))
model = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6,
                           loss_function='Logloss',cat_features=categorical_cols,
                           auto_class_weights='Balanced',thread_count=4,
                           random_seed=100,verbose=False,allow_writing_files=False)
print('Training one final M1 CatBoost fit on',len(fit_rows),'pre-test observations...')
model.fit(prepare_catboost(fit_rows),fit_rows[TARGET_COL].astype(int))
test_scores = model.predict_proba(prepare_catboost(test_rows))[:,1]
test_pred = pd.DataFrame({'row_index':test_rows.index.to_numpy(),
                          'timestamp':test_rows[TIME_COL].to_numpy(),
                          'y_true':test_rows[TARGET_COL].astype(int).to_numpy(),
                          'y_score':test_scores,
                          'amount':pd.to_numeric(test_rows['amt'],errors='raise').to_numpy()})
test_pred['date'] = test_pred['timestamp'].dt.floor('D')
assert len(test_pred)==len(test_rows) and test_pred.row_index.is_unique
assert test_pred['y_score'].between(0,1).all()
model.save_model(str(RESULTS_DIR/'sparkov_final_m1_catboost.cbm'))
val_pred.to_csv(RESULTS_DIR/'sparkov_m1_validation_predictions_with_amount.csv',index=False)
test_pred.to_csv(RESULTS_DIR/'sparkov_m1_test_predictions.csv',index=False)
print('Test PR-AUC:',f'{average_precision_score(test_pred.y_true,test_pred.y_score):.6f}')
print('Test ROC-AUC:',f'{roc_auc_score(test_pred.y_true,test_pred.y_score):.6f}')

Test cutoff: 2020-06-21 00:00:00 | Verified labels strictly before: 2020-06-14 00:00:00
Excluded pre-test unverified rows: 21301
Training one final M1 CatBoost fit on 1275374 pre-test observations...
Test PR-AUC: 0.974357
Test ROC-AUC: 0.999277


## 4. Final model classification and Precision@100 per day

**Precision@100/day** pools the actual fraud count across each day’s top 100 alerts, divided by the total number of top-100 alerts across days (including days with fewer than 100 transactions). This is **not** top 100 over the whole split. Fixed threshold 0.5 is evaluated descriptively, not tuned on test.

In [4]:
def classification_metrics(pred,split):
    y = pred['y_true'].astype(int)
    score = pred['y_score'].astype(float)
    binary = (score>=0.5).astype(int)
    top100 = daily_topk_flags(pred,100)
    return {'split':split,'transactions':len(pred),'fraud_cases':int(y.sum()),
            'pr_auc':average_precision_score(y,score),
            'roc_auc':roc_auc_score(y,score),
            'precision_at_0.5':precision_score(y,binary,zero_division=0),
            'recall_at_0.5':recall_score(y,binary,zero_division=0),
            'precision_at_100_per_day':float(y[top100].sum()/top100.sum()) if top100.sum() else 0.0,
            'top100_alerts':int(top100.sum()),'top100_true_frauds':int(y[top100].sum())}

final_metrics = pd.DataFrame([classification_metrics(val_pred,'validation'),
                              classification_metrics(test_pred,'test')])
display(final_metrics.round(6))
final_metrics.to_csv(RESULTS_DIR/'sparkov_final_classification_metrics.csv',index=False)

,split,transactions,fraud_cases,pr_auc,roc_auc,precision_at_0.5,recall_at_0.5,precision_at_100_per_day,top100_alerts,top100_true_frauds
0,validation,259335,1538,0.989228,0.999875,0.851664,0.981795,0.142315,10800,1537
1,test,555719,2145,0.974357,0.999277,0.758935,0.970163,0.110155,19400,2137


## 5. Locked policy on test: alert volumes, detection and illustrative costs

Apply **BEST_K from validation** unchanged to test. Report TP, FP, FN, TN, precision, recall, review expenditure, missed-fraud amount, total proxy cost, no-alert baseline and percentage savings. Include a separate **K=100/day** comparison for the presentation, without using it to replace the locked policy.

In [5]:
operating_table = pd.DataFrame([
    {'split':'validation',**operating_metrics(val_pred,BEST_K)},
    {'split':'test',**operating_metrics(test_pred,BEST_K)},
])
display(operating_table.round(4))
operating_table.to_csv(RESULTS_DIR/'sparkov_final_test_operational_metrics.csv',index=False)

at100 = pd.DataFrame([{'split':'validation',**operating_metrics(val_pred,100)},
                      {'split':'test',**operating_metrics(test_pred,100)}])
display(at100.round(4))
at100.to_csv(RESULTS_DIR/'sparkov_operational_at_100_per_day.csv',index=False)

# Diagnostic daily alert distribution; dates are evaluated separately.
daily_rows=[]
for split,pred in [('validation',val_pred),('test',test_pred)]:
    flagged = daily_topk_flags(pred,BEST_K)
    for day,group in pred.assign(alert=flagged).groupby('date'):
        daily_rows.append({'split':split,'date':day,'transactions':len(group),
                           'fraud_cases':int(group.y_true.sum()),
                           'alerts':int(group.alert.sum()),
                           'detected_fraud':int((group.alert & group.y_true.astype(bool)).sum())})
daily_alerts=pd.DataFrame(daily_rows)
display(daily_alerts.groupby('split')[['transactions','fraud_cases','alerts','detected_fraud']].agg(['mean','min','max']).round(2))
daily_alerts.to_csv(RESULTS_DIR/'sparkov_daily_alert_breakdown.csv',index=False)

,split,k,days,alerts,tp,fp,fn,tn,precision,recall,review_cost,missed_fraud_amount,total_cost,no_alert_cost,cost_savings,cost_savings_pct
0,validation,50,108,5400,1534,3866,4,253931,0.2841,0.9974,18738.0,852.79,19590.79,816266.27,796675.48,97.6000
1,test,50,194,9700,2130,7570,15,546004,0.2196,0.9930,33659.0,1639.08,35298.08,1133324.68,1098026.60,96.8854


,split,k,days,alerts,tp,fp,fn,tn,precision,recall,review_cost,missed_fraud_amount,total_cost,no_alert_cost,cost_savings,cost_savings_pct
0,validation,100,108,10800,1537,9263,1,248534,0.1423,0.9993,37476.0,20.62,37496.62,816266.27,778769.65,95.4063
1,test,100,194,19400,2137,17263,8,536311,0.1102,0.9963,67318.0,221.37,67539.37,1133324.68,1065785.31,94.0406


transactions             fraud_cases         alerts          \
                   mean   min   max        mean min max   mean min max   
split                                                                    
test            2864.53  1460  6530       11.06   0  40   50.0  50  50   
validation      2401.25  1366  4167       14.24   0  41   50.0  50  50   

           detected_fraud          
                     mean min max  
split                              
test                10.98   0  40  
validation          14.20   0  41

## 6. Test-week performance and final report handoff

Weekly PR-AUC is **descriptive**, not a basis for changing the selected model. A week with no fraud cases has undefined PR-AUC and is recorded as missing.

In [6]:
TEST_START = test_pred['timestamp'].min()
weekly_rows=[]
work=test_pred.copy()
work['week']=((work['timestamp']-TEST_START).dt.total_seconds()//(7*86400)).astype(int)+1
for week,block in work.groupby('week'):
    weekly_rows.append({'week':int(week),'transactions':len(block),
                        'fraud_cases':int(block.y_true.sum()),
                        'pr_auc':average_precision_score(block.y_true,block.y_score)
                                 if block.y_true.sum()>0 else np.nan})
weekly=pd.DataFrame(weekly_rows)
display(weekly.round(6))
weekly.to_csv(RESULTS_DIR/'sparkov_final_test_weekly_prauc.csv',index=False)

print('FINAL MODEL: M1 static CatBoost; validation-selected daily K =',BEST_K)
print('Validation / test classification metrics:')
display(final_metrics[['split','pr_auc','roc_auc','precision_at_0.5','recall_at_0.5','precision_at_100_per_day']].round(6))
print('Locked-policy validation / test operating metrics:')
display(operating_table[['split','k','alerts','tp','fp','fn','precision','recall','review_cost','missed_fraud_amount','total_cost','cost_savings_pct']].round(4))
print('All CSVs saved to:',RESULTS_DIR.resolve())

,week,transactions,fraud_cases,pr_auc
0,1,19982,68,0.962261
1,2,20025,100,0.982127
2,3,20182,83,0.955881
3,4,19777,52,0.963989
4,5,19898,99,0.986590
5,6,19733,64,0.948061
6,7,20023,152,0.982131
7,8,19637,82,0.978237
8,9,19722,59,0.949079
9,10,19605,102,0.992374


FINAL MODEL: M1 static CatBoost; validation-selected daily K = 50
Validation / test classification metrics:


,split,pr_auc,roc_auc,precision_at_0.5,recall_at_0.5,precision_at_100_per_day
0,validation,0.989228,0.999875,0.851664,0.981795,0.142315
1,test,0.974357,0.999277,0.758935,0.970163,0.110155


Locked-policy validation / test operating metrics:


,split,k,alerts,tp,fp,fn,precision,recall,review_cost,missed_fraud_amount,total_cost,cost_savings_pct
0,validation,50,5400,1534,3866,4,0.2841,0.9974,18738.0,852.79,19590.79,97.6000
1,test,50,9700,2130,7570,15,0.2196,0.9930,33659.0,1639.08,35298.08,96.8854


All CSVs saved to: C:\Users\xiang\Downloads\NUS\BT4012\bt4012-credit-card-fraud\results\sparkov_final_test_catboost



## 7. Interpretation and Conclusion

### 7.1 Final Model Performance

The selected M1 static cost-sensitive CatBoost achieved a validation PR-AUC of **0.989228** and a held-out test PR-AUC of **0.974357**, demonstrating strong fraud-ranking performance on unseen transactions. Test ROC-AUC was **0.999277**, while precision and recall at the fixed 0.5 threshold were **0.758935** and **0.970163**, respectively.

Although test PR-AUC decreased by approximately 0.0149 relative to validation, the model maintained strong discriminatory performance. The final model was refitted once using verified pre-test training and validation observations, with its hyperparameters unchanged.

### 7.2 Operational Evaluation

Using the daily alert capacity of **K = 50**, selected exclusively on validation data, the model generated **9,700 test alerts** across 194 days. These identified **2,130 of 2,145 fraudulent transactions**, achieving operational recall of **99.30%** and precision of **21.96%**.

At a higher capacity of **K = 100 alerts/day**, recall increased slightly to **99.63%**, but precision decreased to **11.02%**, requiring twice as many reviews. This highlights the trade-off between fraud detection and operational workload.

Under the illustrative cost assumptions of **$3.47 per alert review** and the full transaction amount for missed fraud, the selected K = 50 policy incurred an estimated total cost of **$35,298.08**, compared with **$1,133,324.68** under a no-alert scenario. This represents **96.89% illustrative cost savings**, rather than verified financial losses prevented.

### 7.3 Temporal Robustness

Weekly test PR-AUC generally remained high, ranging from approximately **0.945 to 0.999** across weeks containing fraud cases. The final week contained no fraud observations, making PR-AUC undefined.

These results suggest that the selected CatBoost model maintained strong ranking performance across the held-out evaluation period, although weekly variation indicates that performance should continue to be monitored.

### 7.4 Conclusion

Overall, M1 static cost-sensitive CatBoost was retained as the final Sparkov model based on its superior validation PR-AUC relative to M3 and M4. Its held-out test performance supports its effectiveness in ranking fraudulent transactions, while the validation-selected daily alert policy demonstrates the potential for high fraud detection under a constrained review capacity.

The operational cost estimates depend on simplified assumptions, and the final test uses a one-time refit with additional verified pre-test observations. Consequently, validation and test scores are not a perfectly controlled comparison of an identical fitted model. The upstream construction of frequency-based features also remains to be verified for temporal consistency.

Nevertheless, the evaluation provides evidence of strong held-out predictive and operational performance under the implemented experimental pipeline.
